In [1]:
from MicroTensorGrad.Engine import MicroTensor
from MicroTensorGrad.Module import Module
from MicroTensorGrad.utils import create_Batches
from MicroTensorGrad.Optimizers import Adam
from MicroTensorGrad.nn import Linear, Tanh
from MicroTensorGrad.Loss import CrossEntropy

In [2]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

In [3]:
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')

In [4]:
mnist["data"].shape

(70000, 784)

In [5]:
mnist["target"].shape

(70000,)

In [6]:
X = mnist["data"]
Y = mnist["target"]

In [7]:
x, x_test, y, y_test = train_test_split(X, Y, test_size=0.2, stratify=Y, random_state=42)

In [8]:
x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.2, stratify=y, random_state=42)

In [9]:
x_train.shape

(44800, 784)

In [10]:
y_train.shape

(44800,)

In [11]:
np.unique(y_train)

array(['0', '1', '2', '3', '4', '5', '6', '7', '8', '9'], dtype=object)

In [12]:
def vectorize_y(y):
    out = []
    for value in y:
        vec = np.zeros((10, ))
        vec[int(value)] = 1
        out.append(vec.copy())
    return np.array(out)

In [13]:
y_train = vectorize_y(y_train)
y_val = vectorize_y(y_val)
y_test = vectorize_y(y_test)

In [14]:
x_train = x_train[..., None]
x_val = x_val[..., None]
x_test = x_test[..., None]

y_train = y_train[..., None]
y_val = y_val[..., None]
y_test = y_test[..., None]

In [15]:
mnist_model = Module(
    components=[
        Linear(784, 512),
        Tanh(),
        Linear(512, 10)
    ]
)

In [16]:
optimizer = Adam(model=mnist_model, lr=0.001)

In [17]:
BatchSize = 1024

In [18]:
x_train_batches, y_train_batches = create_Batches(x_train, y_train, bacthSize=BatchSize)

In [19]:
epochs = 12

In [20]:
x_val_MicroTensor = MicroTensor(x_val)
x_train_MicroTensor = MicroTensor(x_train)

In [21]:
def accuracy(y_true, y_pred):
    y_true1 = y_true.reshape(y_true.shape[0], -1)
    y_pred1 = y_pred.reshape(y_true.shape[0], -1)
    
    return (y_true1.argmax(axis=1) == y_pred1.argmax(axis=1)).sum()/len(y_true1)

In [22]:
best_val_accuracy = -np.inf
best_weights = None

for i in range(epochs):
    MicroTensor.training = True
    for x_batch, y_batch in zip(x_train_batches, y_train_batches):
        x_microTensor = MicroTensor(x_batch)
        topo = mnist_model(x_microTensor)
        loss = CrossEntropy(topo, y_batch)
        loss.backward()
        optimizer.update()
        
        mnist_model.zero_grad()
    
    val_loss = CrossEntropy(mnist_model(x_val_MicroTensor), y_val).topo[-1].array[0]
    train_loss = CrossEntropy(mnist_model(x_train_MicroTensor), y_train).topo[-1].array[0]
    
    MicroTensor.training = False
    val_accuracy = accuracy(y_val, mnist_model(x_val_MicroTensor))
    train_accuracy = accuracy(y_train, mnist_model(x_train_MicroTensor))
    
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        best_weights = mnist_model.BackUp_Weights()
    
    print(f"epoch {i}, train loss : {train_loss}, train accuracy : {train_accuracy} | val loss : {val_loss}, val accuracy : {val_accuracy}")

epoch 0, train loss : 0.4030581890831247, train accuracy : 0.8780580357142858 | val loss : 0.41410584650422355, val accuracy : 0.8720535714285714
epoch 1, train loss : 0.2951382118319254, train accuracy : 0.9122321428571428 | val loss : 0.3074881262764368, val accuracy : 0.9075
epoch 2, train loss : 0.2484748613102562, train accuracy : 0.92453125 | val loss : 0.2651130500260203, val accuracy : 0.9195535714285714
epoch 3, train loss : 0.23332944742360837, train accuracy : 0.9314508928571429 | val loss : 0.24765822370911, val accuracy : 0.9266071428571429
epoch 4, train loss : 0.2121340581829792, train accuracy : 0.9362053571428571 | val loss : 0.22997039234595784, val accuracy : 0.9345535714285714
epoch 5, train loss : 0.2135477030497343, train accuracy : 0.9358705357142857 | val loss : 0.23003217756968444, val accuracy : 0.9336607142857143
epoch 6, train loss : 0.19583015239006582, train accuracy : 0.9400669642857142 | val loss : 0.21751689559268908, val accuracy : 0.933125
epoch 7, tr

In [23]:
mnist_model.Restore_Weights(best_weights)

In [24]:
MicroTensor.training = False

In [25]:
accuracy(y_test, mnist_model(MicroTensor(x_test)))

np.float64(0.9396428571428571)